## Hybrid Search Langchain

In [1]:
# load environment key
from dotenv import load_dotenv
import os

load_dotenv()
api_key = os.getenv("PINECONE_API_KEY")

In [2]:
from langchain_community.retrievers import PineconeHybridSearchRetriever

In [3]:
import os
from pinecone import Pinecone, ServerlessSpec
index_name = "hybrid-search-langchain-pinecone"

# initialize pinecone
pc = Pinecone(api_key=api_key)

# create the index
if index_name not in pc.list_indexes().names():
    pc.create_index(
        name=index_name,
        dimension=384, # dimension of dense vector
        metric="dotproduct", # sparse values only supported for dot product
        spec=ServerlessSpec(
            cloud="aws",
            region="us-east-1"
        )
    )

In [5]:
index = pc.Index(index_name)
index

In [7]:
## Vector embedding and sparse matrix
from langchain_huggingface import HuggingFaceEmbeddings
embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
embeddings

HuggingFaceEmbeddings(model_name='sentence-transformers/all-MiniLM-L6-v2', cache_folder=None, model_kwargs={}, encode_kwargs={}, query_encode_kwargs={}, multi_process=False, show_progress=False)

In [8]:
from pinecone_text.sparse import BM25Encoder

bm25_encoder = BM25Encoder().default()
bm25_encoder

In [9]:
sentences = [
        "In 2023, I visited Paris",
        "In 2022, I visited New York",
        "In 2021, I visited New Orleans",
    ]


## tf idf values on these sentences
bm25_encoder.fit(sentences)

# store the values to a json file
bm25_encoder.dump("bm25_values.json")


## load to your bm25 encoder 
bm25_encoder = BM25Encoder().load("bm25_values.json")

  0%|          | 0/3 [00:00<?, ?it/s]

In [10]:
retriever = PineconeHybridSearchRetriever(
    embeddings=embeddings, sparse_encoder=bm25_encoder, index=index
)

In [11]:
retriever.add_texts(sentences)

  0%|          | 0/1 [00:00<?, ?it/s]

In [15]:
retriever.invoke("What city did i visit ")

[Document(metadata={'score': 0.275548041}, page_content='In 2021, I visited New Orleans'),
 Document(metadata={'score': 0.242202476}, page_content='In 2023, I visited Paris'),
 Document(metadata={'score': 0.259931624}, page_content='In 2022, I visited New York')]